In [1]:
import requests

url = "https://disseminate.stats.swiss/rest/dataflow/CH1.COU/DF_COU_HEALTH_COSTS/1.0.0?references=all"

response = requests.get(url)
response.raise_for_status()

with open("dataflow.xml", "wb") as f:
    f.write(response.content)

from lxml import etree

tree = etree.fromstring(response.content)

pretty_xml = etree.tostring(tree, pretty_print=True, encoding="unicode")

with open("dataflow_pretty.xml", "w", encoding="utf-8") as f:
    f.write(pretty_xml)

print(pretty_xml[:3000])

<message:Structure xmlns:message="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message" xmlns:structure="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure" xmlns:common="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common">
  <message:Header>
    <message:ID>IDREF10800</message:ID>
    <message:Test>false</message:Test>
    <message:Prepared>2026-04-11T09:38:37.1804418+00:00</message:Prepared>
    <message:Sender id="Unknown"/>
    <message:Receiver id="Unknown"/>
  </message:Header>
  <message:Structures>
    <structure:Dataflows>
      <structure:Dataflow id="DF_COU_HEALTH_COSTS" agencyID="CH1.COU" version="1.0.0" isFinal="true">
        <common:Annotations>
          <common:Annotation>
            <common:AnnotationType>NonProductionDataflow</common:AnnotationType>
            <common:AnnotationText xml:lang="en">true</common:AnnotationText>
          </common:Annotation>
          <common:Annotation id="@SDMX">
            <common:AnnotationTitle>P=_T,S=_T,M=_T

In [2]:
from lxml import etree
import pandas as pd

ns = {
    "str": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure",
    "com": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common",
}

tree = etree.parse("dataflow.xml")

rows = []

for dim in tree.xpath("//str:Dimension", namespaces=ns):
    dim_id = dim.get("id")

    concept_ref = dim.find(".//str:ConceptIdentity/Ref", namespaces=ns)
    concept_id = concept_ref.get("id") if concept_ref is not None else None
    concept_scheme = concept_ref.get("maintainableParentID") if concept_ref is not None else None

    codelist_ref = dim.find(".//str:LocalRepresentation/str:Enumeration/Ref", namespaces=ns)
    codelist_id = codelist_ref.get("id") if codelist_ref is not None else None

    rows.append({
        "dimension_id": dim_id,
        "concept_id": concept_id,
        "concept_scheme": concept_scheme,
        "codelist_id": codelist_id,
    })

df_dims = pd.DataFrame(rows)
df_dims

,dimension_id,concept_id,concept_scheme,codelist_id
0,P,P,CS_COU_HEALTH_COSTS,CL_HEALTH_PROVIDER_added_TOTAL
1,S,S,CS_COU_HEALTH_COSTS,CL_HEALTH_SERVICE_added_TOTAL
2,M,M,CS_COU_HEALTH_COSTS,CL_HEALTH_MODE_added_TOTAL
3,F,F,CS_COU_HEALTH_COSTS,CL_HEALTH_FINANCING_added_TOTAL
4,AGE,AGE,CS_COU_HEALTH_COSTS,CL_CLASS_AGE_added_TOTAL
5,GENDER,GENDER,CS_COU_HEALTH_COSTS,CL_SEXE_added_TOTAL
6,CANTON,CANTON,CS_COU_HEALTH_COSTS,CL_HGDE_KT_TOTAL
7,UNIT,UNIT,CS_COU_HEALTH_COSTS,CL_UNIT_MEASURE
8,FREQ,FREQ,CS_COU_HEALTH_COSTS,CL_FREQ
9,NaN,NaN,NaN,NaN


In [3]:
concept_rows = []

for concept in tree.xpath("//str:Concept", namespaces=ns):
    concept_id = concept.get("id")
    names = concept.xpath("./com:Name", namespaces=ns)
    name = names[0].text if names else None

    concept_rows.append({
        "concept_id": concept_id,
        "description": name,
    })

df_concepts = pd.DataFrame(concept_rows)
df_concepts

,concept_id,description
0,TIME_PERIOD,Zeitperiode
1,OBS_VALUE,Observation value (DotStat)
2,DIFF_REGION_STATE,Raumbezugsdatum
3,DIFF_LAST_UPDATE,Aktualisierungsdatum
4,DIFF_EMBARGO_DATE,Publikationsdatum
5,DIFF_DB_STATE,Stand der Datenbank
6,P,Leistungserbringer
7,S,Leistung
8,M,Art der Leistungserbringung
9,F,Finanzierungsregime


In [4]:
df_dict = df_dims.merge(
    df_concepts[["concept_id", "description"]],
    on="concept_id",
    how="left"
)

df_dict

,dimension_id,concept_id,concept_scheme,codelist_id,description
0,P,P,CS_COU_HEALTH_COSTS,CL_HEALTH_PROVIDER_added_TOTAL,Leistungserbringer
1,S,S,CS_COU_HEALTH_COSTS,CL_HEALTH_SERVICE_added_TOTAL,Leistung
2,M,M,CS_COU_HEALTH_COSTS,CL_HEALTH_MODE_added_TOTAL,Art der Leistungserbringung
3,F,F,CS_COU_HEALTH_COSTS,CL_HEALTH_FINANCING_added_TOTAL,Finanzierungsregime
4,AGE,AGE,CS_COU_HEALTH_COSTS,CL_CLASS_AGE_added_TOTAL,Altersklassen
5,GENDER,GENDER,CS_COU_HEALTH_COSTS,CL_SEXE_added_TOTAL,Geschlecht
6,CANTON,CANTON,CS_COU_HEALTH_COSTS,CL_HGDE_KT_TOTAL,Schweizer Kantone
7,UNIT,UNIT,CS_COU_HEALTH_COSTS,CL_UNIT_MEASURE,Masseinheit
8,FREQ,FREQ,CS_COU_HEALTH_COSTS,CL_FREQ,Frequenz
9,NaN,NaN,NaN,NaN,NaN


In [5]:
code_rows = []

for cl in tree.xpath("//str:Codelist", namespaces=ns):
    cl_id = cl.get("id")

    for code in cl.xpath("./str:Code", namespaces=ns):
        code_id = code.get("id")
        names = code.xpath("./com:Name", namespaces=ns)
        name = names[0].text if names else None

        code_rows.append({
            "codelist_id": cl_id,
            "code": code_id,
            "description": name,
        })

df_codes = pd.DataFrame(code_rows)
df_codes

,codelist_id,code,description
0,CL_CLASS_AGE_added_TOTAL,_T,Total
1,CL_CLASS_AGE_added_TOTAL,Y0,0 Jahre
2,CL_CLASS_AGE_added_TOTAL,Y1,1 Jahr
3,CL_CLASS_AGE_added_TOTAL,Y2,2 Jahre
4,CL_CLASS_AGE_added_TOTAL,Y3,3 Jahre
...,...,...,...
1085,CL_UNIT_MULT,5,Hunderttausende
1086,CL_UNIT_MULT,6,Millionen
1087,CL_UNIT_MULT,7,Dutzende Millionen
1088,CL_UNIT_MULT,8,Hunderte von Millionen


In [6]:
df_codes.loc[df_codes['codelist_id'] == "CL_OBS_STATUS"]

,codelist_id,code,description
472,CL_OBS_STATUS,A,Normaler Wert
473,CL_OBS_STATUS,B,Zeitreihenbruch
474,CL_OBS_STATUS,D,Abweichende Definition
475,CL_OBS_STATUS,E,Geschätzter Wert
476,CL_OBS_STATUS,F,Prognostizierter Wert
477,CL_OBS_STATUS,G,Experimenteller Wert
478,CL_OBS_STATUS,H,Fehlender Wert; Feiertag oder Wochenende
479,CL_OBS_STATUS,I,Von einem Empfänger imputierter Wert
480,CL_OBS_STATUS,J,Ausnahmeregelung
481,CL_OBS_STATUS,K,In einer anderen Kategorie enthaltene Daten
